# Policy Scenarios: London Household Electricity Demand Stress Test

This code implements the best model of our group'd ML model (which is thw LightGBM) counterfactual/what-if analysis for the Smart Meters in London project.


## Scenario logic

The scenario set is designed to connect your 2011–2014 half-hourly smart-meter dataset with later London policy themes: building energy efficiency, fuel-poverty targeting, smart meters, half-hourly settlement, and time-of-use flexibility.

we will use six cases:

1. **Baseline observed test period**: original LightGBM prediction on the cached test or winter-test rows.
2. **Cold winter stress**: reduce temperature variables by 2°C and recompute heating/cooling degree days.
3. **Severe cold winter stress**: reduce temperature variables by 3°C and recompute degree days.
4. **Moderate retrofit policy**: reduce historical household-demand proxy variables by 10%.
5. **Cold winter + retrofit**: combine a 2°C cold shock with 10% retrofit savings.
6. **Cold winter + retrofit + ToU flexibility**: combine the cold shock and retrofit with peak-period demand shifting.

## Policy context references

This section provides the policy rationale and references for the what-if design.

- Mayor of London / London City Hall: **Zero carbon London** and the target for London to be net zero carbon by 2030.  
  https://www.london.gov.uk/programmes-strategies/environment-and-climate-change/climate-change/zero-carbon-london

- Mayor of London / London City Hall: **Pathways to Net Zero Carbon by 2030**, including the preferred accelerated green pathway.  
  https://www.london.gov.uk/programmes-strategies/environment-and-climate-change/climate-change/zero-carbon-london/pathways-net-zero-carbon-2030

- Mayor of London / London City Hall: fuel poverty and household energy-efficiency policy context.  
  https://www.london.gov.uk/programmes-strategies/housing-and-land/improving-quality/london-fuel-poverty-action-plan

- Ofgem: **Electricity Retail Market-wide Half-hourly Settlement** decision, which supports more accurate half-hourly settlement and enables more flexible retail products.  
  https://www.ofgem.gov.uk/decision/electricity-retail-market-wide-half-hourly-settlement-decision-and-full-business-case

- Ofgem: discussion of how half-hourly settlement and smart meters support time-of-use tariffs and flexibility.  
  https://www.ofgem.gov.uk/press-release/ofgem-launches-discussion-future-price-cap

In [ ]:
# 0. Imports and notebook-wide configuration

import json
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

# Scenario configuration. Edit these values to test stronger/weaker policy assumptions.
HDD_BASE_C = 15.5
CDD_BASE_C = 22.0

COLD_SHOCK_C = -2.0
SEVERE_COLD_SHOCK_C = -3.0
RETROFIT_SAVING = 0.10        # 10% reduction in historical household demand proxies
STRONG_RETROFIT_SAVING = 0.15 # optional stronger retrofit sensitivity
TOU_PEAK_REDUCTION = 0.07     # 7% reduction in evening peak predictions
TOU_REBOUND_FRACTION = 0.50   # 50% of reduced peak kWh is shifted to overnight/off-peak

PEAK_HOURS = tuple(range(16, 20))  # 16:00-19:30 inclusive as half-hour rows
REBOUND_HOURS = tuple(list(range(22, 24)) + list(range(0, 6)))

# By default, focus the policy stress test on winter rows in the test period.
# If False, the analysis uses the full test set.
USE_WINTER_ONLY = True
WINTER_MONTHS = (12, 1, 2)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# Project paths and scenario configuration.
import json
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
USE_WINTER_ONLY = True
WINTER_MONTHS = [12, 1, 2]
HDD_BASE_C = 15.5
CDD_BASE_C = 22.0


def find_project_root() -> Path:
    try:
        here = Path(__file__).resolve().parent  # type: ignore[name-defined]
    except NameError:
        here = Path.cwd().resolve()

    candidates = [here, *here.parents, Path.cwd().resolve()]
    candidates.extend([
        Path("/content/1.C51-ML-Project"),
        Path("/content/drive/MyDrive/1.C51-ML-Project"),
    ])

    seen = set()
    for cand in candidates:
        try:
            cand = cand.expanduser().resolve()
        except Exception:
            continue
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "README.md").exists() and (cand / "requirements.txt").exists():
            return cand
    return Path.cwd().resolve()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
SMART_DIR = DATA_DIR / "raw" / "smart_meter_data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULTS_DIR / "policy_scenario_figures"

for directory in [PROCESSED_DIR, MODEL_DIR, RESULTS_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Keep CACHE_DIR as a legacy alias for feature tables produced by the main notebook.
CACHE_DIR = PROCESSED_DIR

print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"SMART_DIR     = {SMART_DIR}")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")
print(f"MODEL_DIR     = {MODEL_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")


In [ ]:
# 2. Cache table helpers and feature loading

KNOWN_CAT_COLS = ["icon", "precipType", "Acorn", "Acorn_grouped", "stdorToU"]


def table_path(stem: str) -> Path:
    """Return the first existing cache table path for a stem, or the default parquet path."""
    for ext in (".parquet", ".pkl.gz", ".pkl", ".csv"):
        p = CACHE_DIR / f"{stem}{ext}"
        if p.exists():
            return p
    return CACHE_DIR / f"{stem}.parquet"


def cache_table_exists(stem: str) -> bool:
    return table_path(stem).exists()


def load_table(stem: str, **kwargs) -> pd.DataFrame:
    p = table_path(stem)
    if not p.exists():
        raise FileNotFoundError(f"Cache table not found for '{stem}' in {CACHE_DIR}")
    if p.suffix == ".csv":
        return pd.read_csv(p, **kwargs)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kwargs)
    return pd.read_pickle(p)


def save_table(df: pd.DataFrame, stem: str, prefer_csv: bool = False) -> Path:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if prefer_csv:
        p = CACHE_DIR / f"{stem}.csv"
        df.to_csv(p, index=False)
        return p
    p = CACHE_DIR / f"{stem}.parquet"
    try:
        df.to_parquet(p, index=False)
        return p
    except Exception as e:
        print(f"Parquet write failed for {stem}: {type(e).__name__}: {e}")
        p = CACHE_DIR / f"{stem}.pkl.gz"
        df.to_pickle(p, compression="gzip")
        return p


def load_features() -> pd.DataFrame:
    df = load_table("features_sample500")
    df["tstp"] = pd.to_datetime(df["tstp"], errors="coerce")
    if df["tstp"].isna().any():
        raise ValueError("Some tstp values could not be parsed as datetimes.")
    for c in KNOWN_CAT_COLS:
        if c in df.columns:
            df[c] = df[c].astype("category")
    return df


df_feat = load_features()
print(df_feat.shape)
display(df_feat.head())

In [ ]:
# 3. Load the cached LightGBM model only

try:
    import lightgbm as lgb
except ImportError:
    print("lightgbm not installed. Attempting a quiet pip install...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb

LGB_MODEL_PATH = MODEL_DIR / "lgbm_hh.txt"
LGB_FEATURES_PATH = MODEL_DIR / "lgbm_hh_features.json"

if not LGB_MODEL_PATH.exists() or not LGB_FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Cached LightGBM files are missing. Run the LightGBM section of the main cache pipeline first.\n"
        f"Expected: {LGB_MODEL_PATH}\n"
        f"Expected: {LGB_FEATURES_PATH}"
    )

with open(LGB_FEATURES_PATH, "r") as f:
    lgb_meta = json.load(f)

LGB_NUM_FEATS = lgb_meta.get("num", [])
LGB_CAT_FEATS = lgb_meta.get("cat", [])
LGB_ALL_FEATS = LGB_NUM_FEATS + LGB_CAT_FEATS
BEST_ITERATION = lgb_meta.get("best_iteration", None)

missing = [c for c in LGB_ALL_FEATS if c not in df_feat.columns]
if missing:
    raise KeyError(f"The cached feature table is missing LightGBM features: {missing}")

lgb_model = lgb.Booster(model_file=str(LGB_MODEL_PATH))
print(f"Loaded LightGBM model: {LGB_MODEL_PATH.name}")
print(f"Features: {len(LGB_NUM_FEATS)} numeric + {len(LGB_CAT_FEATS)} categorical")
print(f"Best iteration: {BEST_ITERATION}")

In [ ]:
# 4. Split utilities and policy analysis sample


def get_split_cutoffs(df: pd.DataFrame):
    split_path = CACHE_DIR / "split_metadata.json"
    if split_path.exists():
        meta = json.loads(split_path.read_text())
        return pd.Timestamp(meta["train_cutoff"]), pd.Timestamp(meta["val_cutoff"])
    train_cutoff = pd.Timestamp(df["tstp"].quantile(0.70))
    val_cutoff = pd.Timestamp(df["tstp"].quantile(0.85))
    print("WARNING: split_metadata.json not found. Inferred cutoffs from cached features.")
    return train_cutoff, val_cutoff


def make_split_masks(df: pd.DataFrame):
    train_cutoff, val_cutoff = get_split_cutoffs(df)
    train_mask = df["tstp"] < train_cutoff
    val_mask = (df["tstp"] >= train_cutoff) & (df["tstp"] < val_cutoff)
    test_mask = df["tstp"] >= val_cutoff
    return train_mask, val_mask, test_mask, train_cutoff, val_cutoff

train_mask, val_mask, test_mask, train_cutoff, val_cutoff = make_split_masks(df_feat)

base_df = df_feat.loc[test_mask].copy()
if USE_WINTER_ONLY:
    winter_df = base_df[base_df["tstp"].dt.month.isin(WINTER_MONTHS)].copy()
    if len(winter_df) == 0:
        print("WARNING: no winter rows found in test period; using full test set instead.")
    else:
        base_df = winter_df

print(f"Train cutoff: {train_cutoff}")
print(f"Val cutoff:   {val_cutoff}")
print(f"Policy analysis rows: {len(base_df):,}")
print(f"Households: {base_df['LCLid'].nunique():,}")
print(f"Date range: {base_df['tstp'].min()} -> {base_df['tstp'].max()}")
print(f"Months included: {sorted(base_df['tstp'].dt.month.unique())}")

In [ ]:
# 5. Prediction helper for the cached LightGBM Booster
def prepare_lgb_matrix(df: pd.DataFrame) -> pd.DataFrame:
    X = df[LGB_ALL_FEATS].copy()
    for c in LGB_CAT_FEATS:
        if c in X.columns and X[c].dtype.name != "category":
            X[c] = X[c].astype("category")
    return X


def predict_lgbm(df: pd.DataFrame) -> np.ndarray:
    X = prepare_lgb_matrix(df)
    pred = lgb_model.predict(X, num_iteration=BEST_ITERATION)
    return np.clip(np.asarray(pred, dtype="float32"), 0, None)

# Baseline observed-test prediction.
base_pred = predict_lgbm(base_df)
base_pred_df = base_df[["LCLid", "tstp", "Acorn_grouped", "stdorToU", "household_train_mean", "hdd"]].copy()
base_pred_df["scenario"] = "baseline_observed"
base_pred_df["y_pred_model"] = base_pred
base_pred_df["y_pred"] = base_pred

# Optional diagnostic against observed y for the baseline only.
if "energy" in base_df.columns:
    resid = base_pred - base_df["energy"].to_numpy(dtype="float32")
    print(f"Baseline diagnostic MAE:  {np.mean(np.abs(resid)):.4f}")
    print(f"Baseline diagnostic RMSE: {np.sqrt(np.mean(resid ** 2)):.4f}")
    print(f"Baseline diagnostic bias: {np.mean(resid):+.4f}")

display(base_pred_df.head())

## Counterfactual feature functions

The following functions implement the what-if levers. They modify the model input features, not the original raw smart-meter records.

- The **cold shock** decreases temperature-related features and recomputes `hdd` and `cdd`.
- The **retrofit saving** scales down lagged, rolling, and household baseline demand variables. This represents lower demand after improved insulation and efficiency, not a direct measurement of any single policy scheme.
- The **ToU flexibility** is applied as a post-prediction peak-shaving overlay because simply changing `stdorToU` from `Std` to `ToU` would not be a defensible causal estimate. The overlay reduces evening demand and optionally shifts part of that demand to overnight periods.

In [ ]:
# 6. Counterfactual feature and demand-flexibility functions

DEMAND_PROXY_COLS = [
    "lag_1", "lag_2", "lag_48", "lag_336",
    "rolling_mean_1d", "rolling_std_1d",
    "rolling_mean_7d", "rolling_std_7d",
    "household_train_mean",
]

TEMP_SHIFT_COLS = ["temperature", "apparentTemperature", "dewPoint"]


def recompute_degree_days(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "temperature" in out.columns:
        out["hdd"] = (HDD_BASE_C - out["temperature"]).clip(lower=0).astype("float32")
        out["cdd"] = (out["temperature"] - CDD_BASE_C).clip(lower=0).astype("float32")
    return out


def make_counterfactual_features(
    df: pd.DataFrame,
    temp_shift_c: float = 0.0,
    retrofit_saving: float = 0.0,
) -> pd.DataFrame:
    """Create a LightGBM input table for a policy/weather counterfactual."""
    out = df.copy()

    if temp_shift_c != 0:
        for col in TEMP_SHIFT_COLS:
            if col in out.columns:
                out[col] = (out[col] + temp_shift_c).astype("float32")
        out = recompute_degree_days(out)

    if retrofit_saving != 0:
        if not (0 <= retrofit_saving < 1):
            raise ValueError("retrofit_saving should be a proportion in [0, 1).")
        multiplier = 1.0 - retrofit_saving
        for col in DEMAND_PROXY_COLS:
            if col in out.columns:
                out[col] = (out[col] * multiplier).clip(lower=0).astype("float32")

    # Preserve categorical dtype, which is important for LightGBM.
    for c in LGB_CAT_FEATS:
        if c in out.columns and out[c].dtype.name != "category":
            out[c] = out[c].astype("category")
    return out


def apply_peak_shift(
    pred_df: pd.DataFrame,
    peak_reduction: float = TOU_PEAK_REDUCTION,
    rebound_fraction: float = TOU_REBOUND_FRACTION,
    pred_col: str = "y_pred_model",
) -> pd.DataFrame:
    """Apply a stylised ToU flexibility response to predictions.

    Peak rows are reduced by peak_reduction. A fraction of the saved kWh is distributed
    across overnight/off-peak rows for the same household and day where such rows exist.
    """
    if not (0 <= peak_reduction < 1):
        raise ValueError("peak_reduction should be a proportion in [0, 1).")
    if not (0 <= rebound_fraction <= 1):
        raise ValueError("rebound_fraction should be a proportion in [0, 1].")

    out = pred_df.copy()
    out["y_pred"] = out[pred_col].astype("float32")
    out["date"] = out["tstp"].dt.date
    out["hour"] = out["tstp"].dt.hour

    peak_mask = out["hour"].isin(PEAK_HOURS)
    rebound_mask = out["hour"].isin(REBOUND_HOURS)

    out["saved_kwh"] = 0.0
    out.loc[peak_mask, "saved_kwh"] = out.loc[peak_mask, "y_pred"] * peak_reduction
    out.loc[peak_mask, "y_pred"] = out.loc[peak_mask, "y_pred"] * (1.0 - peak_reduction)

    saved_by_day = out.groupby(["LCLid", "date"], observed=True)["saved_kwh"].sum()
    rebound_counts = out.loc[rebound_mask].groupby(["LCLid", "date"], observed=True).size()

    # Align each rebound row with the available saved energy for its household-date.
    key_index = pd.MultiIndex.from_frame(out.loc[rebound_mask, ["LCLid", "date"]])
    saved_values = saved_by_day.reindex(key_index, fill_value=0).to_numpy()
    counts = rebound_counts.reindex(key_index, fill_value=0).to_numpy()
    additions = np.divide(
        saved_values * rebound_fraction,
        counts,
        out=np.zeros_like(saved_values, dtype="float64"),
        where=counts > 0,
    )
    out.loc[rebound_mask, "y_pred"] = out.loc[rebound_mask, "y_pred"] + additions.astype("float32")

    return out.drop(columns=["saved_kwh"], errors="ignore")

In [ ]:
# 7. Define and run scenarios

SCENARIOS = [
    {
        "scenario": "baseline_observed",
        "description": "Observed test/winter-test conditions.",
        "temp_shift_c": 0.0,
        "retrofit_saving": 0.0,
        "peak_reduction": 0.0,
        "rebound_fraction": 0.0,
    },
    {
        "scenario": "cold_winter_minus_2C",
        "description": "Cold-weather stress: temperature and related variables reduced by 2°C.",
        "temp_shift_c": COLD_SHOCK_C,
        "retrofit_saving": 0.0,
        "peak_reduction": 0.0,
        "rebound_fraction": 0.0,
    },
    {
        "scenario": "severe_cold_minus_3C",
        "description": "Severe cold-weather stress: temperature and related variables reduced by 3°C.",
        "temp_shift_c": SEVERE_COLD_SHOCK_C,
        "retrofit_saving": 0.0,
        "peak_reduction": 0.0,
        "rebound_fraction": 0.0,
    },
    {
        "scenario": "retrofit_10pct",
        "description": "Moderate retrofit: 10% reduction in historical demand proxy variables.",
        "temp_shift_c": 0.0,
        "retrofit_saving": RETROFIT_SAVING,
        "peak_reduction": 0.0,
        "rebound_fraction": 0.0,
    },
    {
        "scenario": "cold_minus_2C_retrofit_10pct",
        "description": "Cold-weather stress with 10% retrofit demand reduction.",
        "temp_shift_c": COLD_SHOCK_C,
        "retrofit_saving": RETROFIT_SAVING,
        "peak_reduction": 0.0,
        "rebound_fraction": 0.0,
    },
    {
        "scenario": "cold_retrofit_tou_flex",
        "description": "Cold-weather stress with 10% retrofit and ToU peak-shifting response.",
        "temp_shift_c": COLD_SHOCK_C,
        "retrofit_saving": RETROFIT_SAVING,
        "peak_reduction": TOU_PEAK_REDUCTION,
        "rebound_fraction": TOU_REBOUND_FRACTION,
    },
    {
        "scenario": "strong_policy_package",
        "description": "Sensitivity case: cold shock with 15% retrofit and stronger 10% peak-shifting response.",
        "temp_shift_c": COLD_SHOCK_C,
        "retrofit_saving": STRONG_RETROFIT_SAVING,
        "peak_reduction": 0.10,
        "rebound_fraction": TOU_REBOUND_FRACTION,
    },
]

scenario_frames = []

for sc in SCENARIOS:
    cf_features = make_counterfactual_features(
        base_df,
        temp_shift_c=sc["temp_shift_c"],
        retrofit_saving=sc["retrofit_saving"],
    )
    y_model = predict_lgbm(cf_features)

    pred_df = cf_features[["LCLid", "tstp", "Acorn_grouped", "stdorToU", "household_train_mean", "hdd"]].copy()
    pred_df["scenario"] = sc["scenario"]
    pred_df["description"] = sc["description"]
    pred_df["temp_shift_c"] = sc["temp_shift_c"]
    pred_df["retrofit_saving"] = sc["retrofit_saving"]
    pred_df["peak_reduction"] = sc["peak_reduction"]
    pred_df["rebound_fraction"] = sc["rebound_fraction"]
    pred_df["y_pred_model"] = y_model
    pred_df["y_pred"] = y_model

    if sc["peak_reduction"] > 0:
        pred_df = apply_peak_shift(
            pred_df,
            peak_reduction=sc["peak_reduction"],
            rebound_fraction=sc["rebound_fraction"],
            pred_col="y_pred_model",
        )

    scenario_frames.append(pred_df)
    print(f"Completed: {sc['scenario']}")

scenario_pred_df = pd.concat(scenario_frames, ignore_index=True)
scenario_pred_df["hour"] = scenario_pred_df["tstp"].dt.hour
scenario_pred_df["date"] = scenario_pred_df["tstp"].dt.date
scenario_pred_df["is_peak"] = scenario_pred_df["hour"].isin(PEAK_HOURS)
scenario_pred_df["is_rebound_period"] = scenario_pred_df["hour"].isin(REBOUND_HOURS)

scenario_path = save_table(scenario_pred_df, "policy_scenario_lightgbm_predictions")
print(f"Saved scenario predictions -> {scenario_path}")
display(scenario_pred_df.head())

In [ ]:
# 8. Scenario summary metrics

def summarize_scenario(pred: pd.DataFrame) -> dict:
    peak = pred[pred["is_peak"]]
    non_peak = pred[~pred["is_peak"]]
    return {
        "n_rows": len(pred),
        "n_households": pred["LCLid"].nunique(),
        "start": pred["tstp"].min(),
        "end": pred["tstp"].max(),
        "total_kwh": pred["y_pred"].sum(),
        "mean_halfhour_kwh": pred["y_pred"].mean(),
        "p95_halfhour_kwh": pred["y_pred"].quantile(0.95),
        "p99_halfhour_kwh": pred["y_pred"].quantile(0.99),
        "max_halfhour_kwh": pred["y_pred"].max(),
        "evening_peak_total_kwh": peak["y_pred"].sum(),
        "evening_peak_mean_kwh": peak["y_pred"].mean(),
        "evening_peak_p95_kwh": peak["y_pred"].quantile(0.95),
        "offpeak_total_kwh": non_peak["y_pred"].sum(),
        "peak_share_of_total_pct": 100 * peak["y_pred"].sum() / pred["y_pred"].sum(),
        "peak_to_average_ratio": pred["y_pred"].quantile(0.95) / pred["y_pred"].mean(),
        "mean_hdd": pred["hdd"].mean(),
    }

summary_rows = []
for scenario, g in scenario_pred_df.groupby("scenario", sort=False):
    row = {"scenario": scenario}
    row.update(summarize_scenario(g))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Add deltas against baseline.
baseline = summary_df.loc[summary_df["scenario"] == "baseline_observed"].iloc[0]
metric_cols = [
    "total_kwh", "mean_halfhour_kwh", "p95_halfhour_kwh", "p99_halfhour_kwh", "max_halfhour_kwh",
    "evening_peak_total_kwh", "evening_peak_mean_kwh", "evening_peak_p95_kwh",
    "offpeak_total_kwh", "peak_share_of_total_pct", "peak_to_average_ratio", "mean_hdd",
]

for col in metric_cols:
    summary_df[f"delta_{col}"] = summary_df[col] - baseline[col]
    denom = baseline[col] if baseline[col] != 0 else np.nan
    summary_df[f"delta_{col}_pct"] = 100 * summary_df[f"delta_{col}"] / denom

summary_csv = RESULTS_DIR / "policy_scenario_lightgbm_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved scenario summary -> {summary_csv}")

display(summary_df[[
    "scenario", "total_kwh", "delta_total_kwh_pct",
    "evening_peak_total_kwh", "delta_evening_peak_total_kwh_pct",
    "p95_halfhour_kwh", "delta_p95_halfhour_kwh_pct",
    "peak_share_of_total_pct", "mean_hdd"
]].round(3))

In [ ]:
# 9. Adequacy test: how much of the cold-weather stress is offset?

def mitigation_ratio(metric: str, scenario_name: str, cold_name: str = "cold_winter_minus_2C") -> float:
    base_val = summary_df.loc[summary_df["scenario"] == "baseline_observed", metric].iloc[0]
    cold_val = summary_df.loc[summary_df["scenario"] == cold_name, metric].iloc[0]
    scen_val = summary_df.loc[summary_df["scenario"] == scenario_name, metric].iloc[0]
    cold_stress = cold_val - base_val
    if np.isclose(cold_stress, 0):
        return np.nan
    return 100 * (cold_val - scen_val) / cold_stress

adequacy_rows = []
for sc in ["cold_minus_2C_retrofit_10pct", "cold_retrofit_tou_flex", "strong_policy_package"]:
    row = {
        "scenario": sc,
        "total_demand_mitigation_vs_cold_pct": mitigation_ratio("total_kwh", sc),
        "evening_peak_mitigation_vs_cold_pct": mitigation_ratio("evening_peak_total_kwh", sc),
        "p95_mitigation_vs_cold_pct": mitigation_ratio("p95_halfhour_kwh", sc),
    }
    row["adequacy_rating"] = "review"
    if (
        row["evening_peak_mitigation_vs_cold_pct"] >= 80
        and summary_df.loc[summary_df["scenario"] == sc, "delta_evening_peak_total_kwh_pct"].iloc[0] <= 5
    ):
        row["adequacy_rating"] = "adequate_under_defined_threshold"
    elif row["evening_peak_mitigation_vs_cold_pct"] >= 50:
        row["adequacy_rating"] = "partially_adequate"
    else:
        row["adequacy_rating"] = "not_adequate_under_defined_threshold"
    adequacy_rows.append(row)

adequacy_df = pd.DataFrame(adequacy_rows)
adequacy_csv = RESULTS_DIR / "policy_scenario_lightgbm_adequacy.csv"
adequacy_df.to_csv(adequacy_csv, index=False)
print(f"Saved adequacy table -> {adequacy_csv}")
display(adequacy_df.round(2))

In [ ]:
# 10. Group-level impact: tariff group, Acorn group, and high-consumption households

# Define high-consumption groups using household_train_mean quartiles.
hh_mean = (
    base_df[["LCLid", "household_train_mean"]]
    .drop_duplicates("LCLid")
    .set_index("LCLid")
)
hh_mean["baseline_use_quartile"] = pd.qcut(
    hh_mean["household_train_mean"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop",
)

scenario_pred_df = scenario_pred_df.merge(
    hh_mean[["baseline_use_quartile"]],
    left_on="LCLid",
    right_index=True,
    how="left",
)

def group_summary(df: pd.DataFrame, group_cols):
    out = (
        df.groupby(["scenario", *group_cols], observed=True)
        .agg(
            n_rows=("y_pred", "size"),
            n_households=("LCLid", "nunique"),
            total_kwh=("y_pred", "sum"),
            mean_halfhour_kwh=("y_pred", "mean"),
            p95_halfhour_kwh=("y_pred", lambda s: s.quantile(0.95)),
            evening_peak_total_kwh=("y_pred", lambda s: s[df.loc[s.index, "is_peak"]].sum()),
        )
        .reset_index()
    )
    return out

by_tariff = group_summary(scenario_pred_df, ["stdorToU"])
by_acorn = group_summary(scenario_pred_df, ["Acorn_grouped"])
by_use_quartile = group_summary(scenario_pred_df, ["baseline_use_quartile"])

group_summary_df = pd.concat(
    [
        by_tariff.assign(group_type="stdorToU").rename(columns={"stdorToU": "group"}),
        by_acorn.assign(group_type="Acorn_grouped").rename(columns={"Acorn_grouped": "group"}),
        by_use_quartile.assign(group_type="baseline_use_quartile").rename(columns={"baseline_use_quartile": "group"}),
    ],
    ignore_index=True,
)

group_csv = RESULTS_DIR / "policy_scenario_lightgbm_group_summary.csv"
group_summary_df.to_csv(group_csv, index=False)
print(f"Saved group summary -> {group_csv}")

display(by_tariff.round(3))
display(by_use_quartile.round(3))

In [ ]:
# 11. Visualisations

plot_metrics = summary_df.set_index("scenario")[[
    "delta_total_kwh_pct",
    "delta_evening_peak_total_kwh_pct",
    "delta_p95_halfhour_kwh_pct",
]].copy()

ax = plot_metrics.plot(kind="bar", figsize=(12, 6))
ax.set_title("Scenario change relative to baseline LightGBM prediction")
ax.set_ylabel("Change vs baseline (%)")
ax.set_xlabel("Scenario")
ax.axhline(0, linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig_path = FIG_DIR / "scenario_delta_bar.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved figure -> {fig_path}")

hourly_profile = (
    scenario_pred_df
    .groupby(["scenario", scenario_pred_df["tstp"].dt.hour], observed=True)["y_pred"]
    .mean()
    .unstack(0)
)
ax = hourly_profile.plot(figsize=(12, 6))
ax.set_title("Mean half-hour demand by hour of day under each scenario")
ax.set_ylabel("Mean predicted kWh per half-hour")
ax.set_xlabel("Hour of day")
plt.tight_layout()
fig_path = FIG_DIR / "hourly_profile_by_scenario.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved figure -> {fig_path}")

peak_share = summary_df.set_index("scenario")[["peak_share_of_total_pct"]]
ax = peak_share.plot(kind="bar", legend=False, figsize=(10, 5))
ax.set_title("Evening-peak share of total predicted demand")
ax.set_ylabel("Peak share of total demand (%)")
ax.set_xlabel("Scenario")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig_path = FIG_DIR / "peak_share_by_scenario.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved figure -> {fig_path}")

In [ ]:
# 12. Auto-generated academic interpretation paragraph

# Extract key values for a reproducible report paragraph.
summary_lookup = summary_df.set_index("scenario")

cold_peak_delta = summary_lookup.loc["cold_winter_minus_2C", "delta_evening_peak_total_kwh_pct"]
cold_total_delta = summary_lookup.loc["cold_winter_minus_2C", "delta_total_kwh_pct"]
combined_peak_delta = summary_lookup.loc["cold_retrofit_tou_flex", "delta_evening_peak_total_kwh_pct"]
combined_total_delta = summary_lookup.loc["cold_retrofit_tou_flex", "delta_total_kwh_pct"]
strong_peak_delta = summary_lookup.loc["strong_policy_package", "delta_evening_peak_total_kwh_pct"]

adequacy_rating = adequacy_df.loc[
    adequacy_df["scenario"] == "cold_retrofit_tou_flex", "adequacy_rating"
].iloc[0]

report_paragraph = f"""
The LightGBM counterfactual results indicate that a 2°C winter cold-shock scenario changes total predicted household electricity demand by {cold_total_delta:.2f}% and evening-peak predicted demand by {cold_peak_delta:.2f}% relative to the observed baseline test period. When a 10% retrofit saving is combined with a stylised time-of-use flexibility response, total predicted demand changes by {combined_total_delta:.2f}% and evening-peak demand changes by {combined_peak_delta:.2f}% relative to baseline. Under the adequacy rule defined in this workbook, the combined policy case is classified as {adequacy_rating.replace('_', ' ')}. This suggests that the adequacy of London-style post-2015 energy policy should be judged not only by average energy reduction but also by its ability to control winter evening peaks. The stronger policy package produces an evening-peak change of {strong_peak_delta:.2f}% relative to baseline, which can be interpreted as a sensitivity case for more ambitious retrofit and flexibility assumptions.
""".strip()

print(report_paragraph)